# Which training examples does a prediction rest on?

This trains a small classifier, scores its training examples for a few queries, and then checks the scores by retraining. It runs on a CPU in a few seconds.

In [1]:
import torch

from culprit import EKFACInfluence
from culprit.evaluate import bootstrap, lds
from culprit.problems import PROBLEMS
from culprit.retrain import RetrainPlan, retrain
from culprit.train import train

A problem bundles a model, its training set, some queries and how to train. This one is a two-layer classifier on overlapping Gaussian clusters.

In [2]:
problem = PROBLEMS["blobs"](0)
task, (train_x, train_y), (query_x, query_y) = problem.task, problem.train, problem.queries
run = train(task, train_x, train_y, problem.config)
print(f"trained for {len(run.losses)} steps, final loss {run.losses[-10:].mean():.3f}")

trained for 300 steps, final loss 0.205


Every method has the same two calls. `scores[q, i]` estimates how much including training example `i` raised the model's log-probability of the right answer for query `q`.

In [3]:
scores = EKFACInfluence(task, damping=1e-2).fit(train_x, train_y).score(query_x, query_y)
print(tuple(scores.shape))

(100, 500)


In [4]:
query = 0
order = scores[query].argsort(descending=True)
print(f"query {query} has label {int(query_y[query])}")
for name, picks in (("most helpful", order[:3]), ("most harmful", order[-3:])):
    for index in picks.tolist():
        print(
            f"  {name}: example {index:3d}, label {int(train_y[index])}, score {scores[query, index]:+.2e}"
        )

query 0 has label 0
  most helpful: example 254, label 0, score +6.19e-04
  most helpful: example 258, label 0, score +4.77e-04
  most helpful: example 348, label 0, score +3.67e-04
  most harmful: example 257, label 1, score -5.73e-04
  most harmful: example 182, label 1, score -9.06e-04
  most harmful: example  41, label 1, score -1.08e-03


The examples that help most share the query's label, and the ones that hurt most do not. That is reassuring and proves nothing. The check is to retrain: 64 models, each on a random half of the training set, recording what each says about every query.

In [5]:
retrained = retrain(
    task, problem.train, problem.queries, problem.config, RetrainPlan(models=64, together=64)
)
print(tuple(retrained.masks.shape), tuple(retrained.measurements.shape))

(64, 500) (64, 100)


If the scores are right, a model's answer is predicted by the sum of the scores of the examples it was trained on. The linear datamodeling score is the rank correlation between the two across the retrained models, for each query. As a control, the same scores are assigned to the wrong examples.

In [6]:
per_query = lds(scores, retrained.masks, retrained.measurements)
print("scores:           ", bootstrap(per_query))
shuffled = scores[:, torch.randperm(scores.shape[1], generator=torch.Generator().manual_seed(0))]
print("shuffled scores:  ", bootstrap(lds(shuffled, retrained.masks, retrained.measurements)))

scores:            0.589 [0.556, 0.619]
shuffled scores:   -0.009 [-0.035, 0.015]
